In [2]:
# Cell 1: Setup paths, imports, and run configuration

from pathlib import Path
from datetime import datetime, timezone
import json
import math
import warnings

import numpy as np
import rasterio

warnings.filterwarnings("ignore", category=RuntimeWarning)

# -------------------------------------------------------------------
# ROOTS
# -------------------------------------------------------------------
# Update this only if your project root is different
PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RASTERS_DIR = PROJECT_ROOT / "rasters"

# Raw inputs
CMIP6_SCENARIOS_DIR = DATA_DIR / "scenarios"
ERA5_DIR = DATA_DIR / "simulated" / "era5_land"

# Reference variable
ERA5_D2M_DIR = ERA5_DIR / "d2m"

# Outputs
CORRECTED_CMIP6_DIR = RASTERS_DIR / "corrected" / "cmip6"
ARTIFACTS_DIR = DATA_DIR / "artifacts"
MANIFESTS_DIR = ARTIFACTS_DIR / "manifests"

# -------------------------------------------------------------------
# RUN CONFIG
# -------------------------------------------------------------------
SCENARIOS = ["ssp245", "ssp370", "ssp585"]
HIST_START = 1981
HIST_END = 2014
FUTURE_START = 2026
FUTURE_END = 2050

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"huss_d2m_{RUN_TS}"

# -------------------------------------------------------------------
# ENSURE OUTPUT FOLDERS EXIST
# -------------------------------------------------------------------
for scenario in SCENARIOS:
    (CORRECTED_CMIP6_DIR / scenario / "d2m").mkdir(parents=True, exist_ok=True)

MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# HELPERS
# -------------------------------------------------------------------
def yyyymm_from_name(path: Path) -> str | None:
    """
    Extract YYYYMM from filenames like:
    huss_202601.tif
    d2m_198103.tif
    d2m_corrected_202601.tif
    """
    stem = path.stem
    parts = stem.split("_")
    for part in reversed(parts):
        if len(part) == 6 and part.isdigit():
            return part
    return None


def list_tifs(folder: Path) -> list[Path]:
    if not folder.exists():
        return []
    return sorted(folder.glob("*.tif"))


def index_by_yyyymm(folder: Path) -> dict[str, Path]:
    out = {}
    for fp in list_tifs(folder):
        key = yyyymm_from_name(fp)
        if key is not None:
            out[key] = fp
    return out


def year_from_yyyymm(yyyymm: str) -> int:
    return int(yyyymm[:4])


def filter_period(index: dict[str, Path], start_year: int, end_year: int) -> dict[str, Path]:
    return {
        ym: fp
        for ym, fp in index.items()
        if start_year <= year_from_yyyymm(ym) <= end_year
    }


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


print("=" * 60)
print("HUSS -> D2M SETUP")
print("=" * 60)
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"ERA5_D2M_DIR : {ERA5_D2M_DIR}")
print(f"CMIP6_DIR    : {CMIP6_SCENARIOS_DIR}")
print(f"OUTPUT_DIR   : {CORRECTED_CMIP6_DIR}")
print(f"RUN_ID       : {RUN_ID}")
print(f"SCENARIOS    : {SCENARIOS}")
print(f"HIST PERIOD  : {HIST_START}-{HIST_END}")
print(f"FUTURE PERIOD: {FUTURE_START}-{FUTURE_END}")

HUSS -> D2M SETUP
PROJECT_ROOT : C:\Projects\Infer RozviDrought
ERA5_D2M_DIR : C:\Projects\Infer RozviDrought\data\simulated\era5_land\d2m
CMIP6_DIR    : C:\Projects\Infer RozviDrought\data\scenarios
OUTPUT_DIR   : C:\Projects\Infer RozviDrought\rasters\corrected\cmip6
RUN_ID       : huss_d2m_20260321T113457Z
SCENARIOS    : ['ssp245', 'ssp370', 'ssp585']
HIST PERIOD  : 1981-2014
FUTURE PERIOD: 2026-2050


In [5]:
# Cell 2 (redo): Index NetCDF huss files per scenario

from pathlib import Path

scenario_huss_nc = {}

print("=" * 60)
print("CMIP6 HUSS NETCDF FILES")
print("=" * 60)

for scenario in SCENARIOS:
    huss_dir = CMIP6_SCENARIOS_DIR / scenario / "huss"

    nc_files = sorted(huss_dir.glob("*.nc"))

    scenario_huss_nc[scenario] = nc_files

    print("\n" + "-" * 60)
    print(f"SCENARIO: {scenario}")
    print("-" * 60)
    print(f"Huss directory : {huss_dir}")
    print(f"NetCDF files   : {len(nc_files)}")

    for fp in nc_files:
        print(fp.name)

# Save simple manifest for traceability
nc_index_summary = {
    "run_id": RUN_ID,
    "type": "huss_nc_inventory",
    "scenarios": {
        s: [str(p) for p in scenario_huss_nc[s]]
        for s in SCENARIOS
    }
}

save_json(
    nc_index_summary,
    MANIFESTS_DIR / f"{RUN_ID}_huss_nc_inventory.json"
)

print("\nSaved NetCDF inventory manifest.")

CMIP6 HUSS NETCDF FILES

------------------------------------------------------------
SCENARIO: ssp245
------------------------------------------------------------
Huss directory : C:\Projects\Infer RozviDrought\data\scenarios\ssp245\huss
NetCDF files   : 2
huss_Amon_MPI-ESM1-2-LR_ssp245_r1i1p1f1_gn_20260116-20501216.nc
huss_MPI-ESM1-2-LR_ssp245_2026_2050.nc

------------------------------------------------------------
SCENARIO: ssp370
------------------------------------------------------------
Huss directory : C:\Projects\Infer RozviDrought\data\scenarios\ssp370\huss
NetCDF files   : 2
huss_Amon_MPI-ESM1-2-LR_ssp370_r1i1p1f1_gn_20260116-20501216.nc
huss_MPI-ESM1-2-LR_ssp370_2026_2050.nc

------------------------------------------------------------
SCENARIO: ssp585
------------------------------------------------------------
Huss directory : C:\Projects\Infer RozviDrought\data\scenarios\ssp585\huss
NetCDF files   : 2
huss_Amon_MPI-ESM1-2-LR_ssp585_r1i1p1f1_gn_20260116-20501216.nc
huss

In [6]:
# Cell 3: Inspect one sample huss NetCDF file

import xarray as xr

# pick one sample file
sample_scenario = "ssp245"
sample_nc = scenario_huss_nc[sample_scenario][0]

print("=" * 60)
print("SAMPLE HUSS NETCDF INSPECTION")
print("=" * 60)
print(f"Scenario   : {sample_scenario}")
print(f"File       : {sample_nc}")

ds = xr.open_dataset(sample_nc)

print("\nDataset summary:")
print(ds)

print("\nVariables:")
for name, da in ds.data_vars.items():
    print(f"- {name}: dims={da.dims}, shape={da.shape}, units={da.attrs.get('units')}")

print("\nCoordinates:")
for name, da in ds.coords.items():
    print(f"- {name}: dims={da.dims}, shape={da.shape}")

# inspect the huss variable directly
huss_var = "huss"
da = ds[huss_var]

print("\nHUSS DETAILS")
print(f"name        : {huss_var}")
print(f"dims        : {da.dims}")
print(f"shape       : {da.shape}")
print(f"units       : {da.attrs.get('units')}")
print(f"long_name   : {da.attrs.get('long_name')}")
print(f"time start  : {str(ds['time'].values[0])}")
print(f"time end    : {str(ds['time'].values[-1])}")
print(f"n_time      : {ds.sizes.get('time')}")

# quick numeric preview from first timestep
arr0 = da.isel(time=0).values
print("\nFIRST TIMESTEP STATS")
print(f"min         : {np.nanmin(arr0)}")
print(f"max         : {np.nanmax(arr0)}")
print(f"mean        : {np.nanmean(arr0)}")

ds.close()

SAMPLE HUSS NETCDF INSPECTION
Scenario   : ssp245
File       : C:\Projects\Infer RozviDrought\data\scenarios\ssp245\huss\huss_Amon_MPI-ESM1-2-LR_ssp245_r1i1p1f1_gn_20260116-20501216.nc

Dataset summary:
<xarray.Dataset> Size: 24MB
Dimensions:    (time: 300, bnds: 2, lat: 96, lon: 192)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2026-01-16T12:00:00 ... 2050-12-16T1...
  * lat        (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
  * lon        (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 354.4 356.2 358.1
    height     float64 8B ...
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) datetime64[ns] 5kB ...
    lat_bnds   (time, lat, bnds) float64 461kB ...
    lon_bnds   (time, lon, bnds) float64 922kB ...
    huss       (time, lat, lon) float32 22MB ...
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ScenarioMIP
    branch_method:          standard
    branch_time_in_child:   60265.0
    b

In [9]:
# Cell 4 (robust): Use only valid CMIP6 NetCDF files

import xarray as xr
import pandas as pd

def nc_time_to_yyyymm(time_values):
    return [f"{t.year}{t.month:02d}" for t in pd.to_datetime(time_values)]

scenario_huss_month_index = {}

print("=" * 60)
print("HUSS MONTH INDEX (VALID FILES)")
print("=" * 60)

for scenario in SCENARIOS:
    huss_dir = CMIP6_SCENARIOS_DIR / scenario / "huss"

    # keep only official CMIP6 files
    nc_files = sorted(
        f for f in huss_dir.glob("*.nc")
        if "Amon" in f.name
    )

    month_map = {}

    for nc_path in nc_files:
        print(f"\nOpening: {nc_path.name}")

        ds = xr.open_dataset(
            nc_path,
            engine="netcdf4",
            decode_times=True
        )

        months = nc_time_to_yyyymm(ds["time"].values)

        for i, ym in enumerate(months):
            month_map[ym] = {
                "nc_path": str(nc_path),
                "time_idx": i,
            }

        ds.close()

    scenario_huss_month_index[scenario] = month_map

    print("\n" + "-" * 60)
    print(f"SCENARIO: {scenario}")
    print(f"Months indexed : {len(month_map)}")

    if month_map:
        keys = sorted(month_map.keys())
        print(f"First month    : {keys[0]}")
        print(f"Last month     : {keys[-1]}")

HUSS MONTH INDEX (VALID FILES)

Opening: huss_Amon_MPI-ESM1-2-LR_ssp245_r1i1p1f1_gn_20260116-20501216.nc

------------------------------------------------------------
SCENARIO: ssp245
Months indexed : 300
First month    : 202601
Last month     : 205012

Opening: huss_Amon_MPI-ESM1-2-LR_ssp370_r1i1p1f1_gn_20260116-20501216.nc

------------------------------------------------------------
SCENARIO: ssp370
Months indexed : 300
First month    : 202601
Last month     : 205012

Opening: huss_Amon_MPI-ESM1-2-LR_ssp585_r1i1p1f1_gn_20260116-20501216.nc

------------------------------------------------------------
SCENARIO: ssp585
Months indexed : 300
First month    : 202601
Last month     : 205012


In [10]:
# Cell 5: Inspect one ERA5 d2m raster and compare its grid to one CMIP6 huss slice

import rasterio
import xarray as xr
from rasterio.transform import array_bounds

# ------------------------------------------------------------
# Pick one ERA5 d2m sample raster
# ------------------------------------------------------------
era5_sample_month = sorted(era5_d2m_index.keys())[0]
era5_sample_path = era5_d2m_index[era5_sample_month]

print("=" * 60)
print("ERA5 D2M SAMPLE")
print("=" * 60)
print(f"Month : {era5_sample_month}")
print(f"File  : {era5_sample_path}")

with rasterio.open(era5_sample_path) as src:
    era5_meta = src.meta.copy()
    era5_crs = src.crs
    era5_transform = src.transform
    era5_width = src.width
    era5_height = src.height
    era5_nodata = src.nodata
    era5_bounds = src.bounds

    arr = src.read(1)
    print(f"CRS       : {era5_crs}")
    print(f"Shape     : ({era5_height}, {era5_width})")
    print(f"Nodata    : {era5_nodata}")
    print(f"Bounds    : {era5_bounds}")
    print(f"Min       : {np.nanmin(arr)}")
    print(f"Max       : {np.nanmax(arr)}")
    print(f"Mean      : {np.nanmean(arr)}")

# ------------------------------------------------------------
# Pick one CMIP6 huss sample slice
# ------------------------------------------------------------
sample_scenario = "ssp245"
sample_month = sorted(scenario_huss_month_index[sample_scenario].keys())[0]
sample_info = scenario_huss_month_index[sample_scenario][sample_month]
sample_nc_path = sample_info["nc_path"]
sample_time_idx = sample_info["time_idx"]

print("\n" + "=" * 60)
print("CMIP6 HUSS SAMPLE")
print("=" * 60)
print(f"Scenario  : {sample_scenario}")
print(f"Month     : {sample_month}")
print(f"NC file   : {sample_nc_path}")
print(f"Time idx  : {sample_time_idx}")

ds = xr.open_dataset(sample_nc_path, engine="netcdf4")
da = ds["huss"].isel(time=sample_time_idx)

huss_arr = da.values
lat = ds["lat"].values
lon = ds["lon"].values

print(f"Shape     : {huss_arr.shape}")
print(f"Lat range : {lat.min()} to {lat.max()}")
print(f"Lon range : {lon.min()} to {lon.max()}")
print(f"Min       : {np.nanmin(huss_arr)}")
print(f"Max       : {np.nanmax(huss_arr)}")
print(f"Mean      : {np.nanmean(huss_arr)}")

ds.close()

print("\n" + "=" * 60)
print("GRID COMPARISON SUMMARY")
print("=" * 60)
print(f"ERA5 grid      : {era5_height} x {era5_width}")
print(f"CMIP6 grid     : {huss_arr.shape[0]} x {huss_arr.shape[1]}")
print("Conclusion     : CMIP6 huss must be converted then reprojected/resampled to ERA5 d2m grid")

ERA5 D2M SAMPLE
Month : 198109
File  : C:\Projects\Infer RozviDrought\data\simulated\era5_land\d2m\d2m_198109.tif
CRS       : EPSG:4326
Shape     : (70, 80)
Nodata    : None
Bounds    : BoundingBox(left=25.100114814474466, bottom=-22.500102921341654, right=33.100151408729275, top=-15.500070901368694)
Min       : 275.1539611816406
Max       : 290.0555114746094
Mean      : 280.9493713378906

CMIP6 HUSS SAMPLE
Scenario  : ssp245
Month     : 202601
NC file   : C:\Projects\Infer RozviDrought\data\scenarios\ssp245\huss\huss_Amon_MPI-ESM1-2-LR_ssp245_r1i1p1f1_gn_20260116-20501216.nc
Time idx  : 0
Shape     : (96, 192)
Lat range : -88.57216851400727 to 88.57216851400727
Lon range : 0.0 to 358.125
Min       : 0.0001531335583422333
Max       : 0.020009109750390053
Mean      : 0.0069405254907906055

GRID COMPARISON SUMMARY
ERA5 grid      : 70 x 80
CMIP6 grid     : 96 x 192
Conclusion     : CMIP6 huss must be converted then reprojected/resampled to ERA5 d2m grid


In [11]:
# Cell 6: Convert one sample CMIP6 huss slice to dewpoint (K), then reproject to ERA5 grid

import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling

# ------------------------------------------------------------
# Physics helpers
# ------------------------------------------------------------
def specific_humidity_to_vapor_pressure(q, p_pa=101325.0):
    """
    Convert specific humidity q (kg/kg) to vapor pressure e (Pa)
    using:
        q = 0.622 e / (p - 0.378 e)
    rearranged to:
        e = q p / (0.622 + 0.378 q)
    """
    q = np.asarray(q, dtype=np.float64)
    e = (q * p_pa) / (0.622 + 0.378 * q)
    return e

def vapor_pressure_to_dewpoint_kelvin(e_pa):
    """
    Convert vapor pressure (Pa) to dewpoint temperature (K)
    Magnus-style form:
        Td(C) = 243.5 * ln(e/611.2) / (17.67 - ln(e/611.2))
    """
    e_pa = np.asarray(e_pa, dtype=np.float64)
    e_pa = np.clip(e_pa, 1.0, None)  # avoid log(0)
    ln_ratio = np.log(e_pa / 611.2)
    td_c = (243.5 * ln_ratio) / (17.67 - ln_ratio)
    td_k = td_c + 273.15
    return td_k

# ------------------------------------------------------------
# Get one sample huss slice
# ------------------------------------------------------------
sample_scenario = "ssp245"
sample_month = "202601"
sample_info = scenario_huss_month_index[sample_scenario][sample_month]

nc_path = sample_info["nc_path"]
time_idx = sample_info["time_idx"]

ds = xr.open_dataset(nc_path, engine="netcdf4")
q = ds["huss"].isel(time=time_idx).values.astype(np.float32)
lat = ds["lat"].values
lon = ds["lon"].values
ds.close()

# Convert 0..360 lon to -180..180, then sort
lon_180 = np.where(lon > 180.0, lon - 360.0, lon)
sort_idx = np.argsort(lon_180)
lon_180 = lon_180[sort_idx]
q = q[:, sort_idx]

# Flip latitude to north->south if needed for raster writing
if lat[0] < lat[-1]:
    lat_desc = lat[::-1]
    q = q[::-1, :]
else:
    lat_desc = lat.copy()

# ------------------------------------------------------------
# Convert q -> e -> Td(K)
# ------------------------------------------------------------
td_k_src = vapor_pressure_to_dewpoint_kelvin(
    specific_humidity_to_vapor_pressure(q, p_pa=101325.0)
).astype(np.float32)

print("=" * 60)
print("SOURCE CONVERSION STATS")
print("=" * 60)
print(f"Sample month : {sample_month}")
print(f"Input q min  : {np.nanmin(q)}")
print(f"Input q max  : {np.nanmax(q)}")
print(f"Td(K) min    : {np.nanmin(td_k_src)}")
print(f"Td(K) max    : {np.nanmax(td_k_src)}")
print(f"Td(K) mean   : {np.nanmean(td_k_src)}")

# ------------------------------------------------------------
# Build source transform from CMIP6 lat/lon
# ------------------------------------------------------------
lon_res = float(np.abs(np.diff(lon_180).mean()))
lat_res = float(np.abs(np.diff(lat_desc).mean()))

src_left = float(lon_180.min() - lon_res / 2.0)
src_right = float(lon_180.max() + lon_res / 2.0)
src_bottom = float(lat_desc.min() - lat_res / 2.0)
src_top = float(lat_desc.max() + lat_res / 2.0)

src_transform = from_bounds(
    src_left, src_bottom, src_right, src_top,
    td_k_src.shape[1], td_k_src.shape[0]
)

# ------------------------------------------------------------
# Reproject to ERA5 d2m grid
# ------------------------------------------------------------
with rasterio.open(era5_sample_path) as ref:
    dst_shape = (ref.height, ref.width)
    dst_transform = ref.transform
    dst_crs = ref.crs
    ref_arr = ref.read(1).astype(np.float32)

td_k_dst = np.full(dst_shape, np.nan, dtype=np.float32)

reproject(
    source=td_k_src,
    destination=td_k_dst,
    src_transform=src_transform,
    src_crs="EPSG:4326",
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.bilinear,
    src_nodata=None,
    dst_nodata=np.nan,
)

print("\n" + "=" * 60)
print("REPROJECTED TO ERA5 GRID")
print("=" * 60)
print(f"Output shape : {td_k_dst.shape}")
print(f"Min          : {np.nanmin(td_k_dst)}")
print(f"Max          : {np.nanmax(td_k_dst)}")
print(f"Mean         : {np.nanmean(td_k_dst)}")

# Optional quick comparison against ERA5 sample raster
print("\n" + "=" * 60)
print("QUICK COMPARISON VS ERA5 SAMPLE MONTH")
print("=" * 60)
print(f"ERA5 mean    : {np.nanmean(ref_arr)}")
print(f"CMIP6 Td mean: {np.nanmean(td_k_dst)}")
print(f"Mean delta   : {np.nanmean(td_k_dst) - np.nanmean(ref_arr)}")

SOURCE CONVERSION STATS
Sample month : 202601
Input q min  : 0.0001531335583422333
Input q max  : 0.020009109750390053
Td(K) min    : 235.82583618164062
Td(K) max    : 298.427978515625
Td(K) mean   : 274.23687744140625

REPROJECTED TO ERA5 GRID
Output shape : (70, 80)
Min          : 293.8247985839844
Max          : 295.40203857421875
Mean         : 294.5806579589844

QUICK COMPARISON VS ERA5 SAMPLE MONTH
ERA5 mean    : 280.9493713378906
CMIP6 Td mean: 294.5806579589844
Mean delta   : 13.63128662109375


In [12]:
# Cell 7: Same-month historical comparison (core validation step)

import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling

# ------------------------------------------------------------
# Reuse physics functions from previous cell
# ------------------------------------------------------------
def specific_humidity_to_vapor_pressure(q, p_pa=101325.0):
    return (q * p_pa) / (0.622 + 0.378 * q)

def vapor_pressure_to_dewpoint_kelvin(e_pa):
    e_pa = np.clip(e_pa, 1.0, None)
    ln_ratio = np.log(e_pa / 611.2)
    td_c = (243.5 * ln_ratio) / (17.67 - ln_ratio)
    return td_c + 273.15

# ------------------------------------------------------------
# Pick a historical overlap month
# (use first ERA5 historical month)
# ------------------------------------------------------------
hist_month = sorted(era5_d2m_hist.keys())[0]

print("=" * 60)
print("HISTORICAL SAME-MONTH COMPARISON")
print("=" * 60)
print(f"Month selected : {hist_month}")

# ------------------------------------------------------------
# Load ERA5 reference raster
# ------------------------------------------------------------
era5_path = era5_d2m_hist[hist_month]

with rasterio.open(era5_path) as ref:
    ref_arr = ref.read(1).astype(np.float32)
    dst_shape = (ref.height, ref.width)
    dst_transform = ref.transform
    dst_crs = ref.crs

print(f"ERA5 mean : {np.nanmean(ref_arr)}")

# ------------------------------------------------------------
# Load matching CMIP6 historical huss
# (NOTE: assumes historical NetCDF exists in data/historical/cmip6)
# ------------------------------------------------------------
hist_nc = Path(
    r"C:\Projects\Infer RozviDrought\data\historical\cmip6\huss"
).glob("*Amon*.nc")

hist_nc = sorted(hist_nc)[0]

ds = xr.open_dataset(hist_nc, engine="netcdf4")

time_values = pd.to_datetime(ds["time"].values)
match_idx = np.where(
    (time_values.year == int(hist_month[:4])) &
    (time_values.month == int(hist_month[4:]))
)[0][0]

q = ds["huss"].isel(time=match_idx).values
lat = ds["lat"].values
lon = ds["lon"].values
ds.close()

# ------------------------------------------------------------
# Convert lon to -180..180
# ------------------------------------------------------------
lon_180 = np.where(lon > 180.0, lon - 360.0, lon)
sort_idx = np.argsort(lon_180)

lon_180 = lon_180[sort_idx]
q = q[:, sort_idx]

# flip latitude if needed
if lat[0] < lat[-1]:
    lat = lat[::-1]
    q = q[::-1, :]

# ------------------------------------------------------------
# Convert to dewpoint
# ------------------------------------------------------------
td_src = vapor_pressure_to_dewpoint_kelvin(
    specific_humidity_to_vapor_pressure(q)
).astype(np.float32)

# ------------------------------------------------------------
# Build transform
# ------------------------------------------------------------
lon_res = float(np.abs(np.diff(lon_180).mean()))
lat_res = float(np.abs(np.diff(lat).mean()))

src_transform = from_bounds(
    lon_180.min() - lon_res / 2,
    lat.min() - lat_res / 2,
    lon_180.max() + lon_res / 2,
    lat.max() + lat_res / 2,
    td_src.shape[1],
    td_src.shape[0]
)

# ------------------------------------------------------------
# Reproject to ERA5 grid
# ------------------------------------------------------------
td_dst = np.full(dst_shape, np.nan, dtype=np.float32)

reproject(
    source=td_src,
    destination=td_dst,
    src_transform=src_transform,
    src_crs="EPSG:4326",
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.bilinear,
)

print("\n" + "=" * 60)
print("RESULT")
print("=" * 60)

print(f"CMIP6 Td mean : {np.nanmean(td_dst)}")
print(f"ERA5 mean     : {np.nanmean(ref_arr)}")
print(f"Mean delta    : {np.nanmean(td_dst) - np.nanmean(ref_arr)}")

HISTORICAL SAME-MONTH COMPARISON
Month selected : 198109
ERA5 mean : 280.9493713378906

RESULT
CMIP6 Td mean : 279.74896240234375
ERA5 mean     : 280.9493713378906
Mean delta    : -1.200408935546875


In [13]:
# Cell 8: Build d2m training dataset and save to artifacts/datasets

import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from pathlib import Path
from datetime import datetime, timezone
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling

# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

OUTPUT_DATASET_PATH = (
    DATA_DIR
    / "artifacts"
    / "datasets"
    / f"cmip6_d2m_training_dataset_{timestamp}.parquet"
)

OUTPUT_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

records = []

# ------------------------------------------------------------
# Use historical months only
# ------------------------------------------------------------
hist_months = sorted(era5_d2m_hist.keys())

print("=" * 60)
print("BUILDING D2M TRAINING DATASET")
print("=" * 60)
print(f"Months to process : {len(hist_months)}")

for month in hist_months:

    era5_path = era5_d2m_hist[month]

    # Load ERA5
    with rasterio.open(era5_path) as ref:
        era5_arr = ref.read(1).astype(np.float32)

        dst_shape = (ref.height, ref.width)
        dst_transform = ref.transform
        dst_crs = ref.crs

    # Load CMIP6 historical
    hist_nc_files = sorted(
        Path(
            r"C:\Projects\Infer RozviDrought\data\historical\cmip6\huss"
        ).glob("*Amon*.nc")
    )

    ds = xr.open_dataset(hist_nc_files[0], engine="netcdf4")

    time_values = pd.to_datetime(ds["time"].values)

    idx = np.where(
        (time_values.year == int(month[:4])) &
        (time_values.month == int(month[4:]))
    )[0][0]

    q = ds["huss"].isel(time=idx).values
    lat = ds["lat"].values
    lon = ds["lon"].values

    ds.close()

    # Convert lon
    lon_180 = np.where(lon > 180, lon - 360, lon)
    sort_idx = np.argsort(lon_180)

    lon_180 = lon_180[sort_idx]
    q = q[:, sort_idx]

    if lat[0] < lat[-1]:
        lat = lat[::-1]
        q = q[::-1, :]

    # Convert to dewpoint
    e = (q * 101325.0) / (0.622 + 0.378 * q)

    ln_ratio = np.log(np.clip(e, 1.0, None) / 611.2)

    td_c = (243.5 * ln_ratio) / (17.67 - ln_ratio)

    td_k = td_c + 273.15

    # Build transform
    lon_res = float(np.abs(np.diff(lon_180).mean()))
    lat_res = float(np.abs(np.diff(lat).mean()))

    src_transform = from_bounds(
        lon_180.min() - lon_res / 2,
        lat.min() - lat_res / 2,
        lon_180.max() + lon_res / 2,
        lat.max() + lat_res / 2,
        td_k.shape[1],
        td_k.shape[0]
    )

    td_dst = np.full(dst_shape, np.nan, dtype=np.float32)

    reproject(
        source=td_k.astype(np.float32),
        destination=td_dst,
        src_transform=src_transform,
        src_crs="EPSG:4326",
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
    )

    pixel_ids = np.arange(td_dst.size)

    df = pd.DataFrame({
        "yyyymm": month,
        "pixel_id": pixel_ids,
        "d2m_cmip6": td_dst.flatten(),
        "d2m_era5": era5_arr.flatten(),
    })

    records.append(df)

    print(f"Processed month : {month}")

# ------------------------------------------------------------
# Save dataset
# ------------------------------------------------------------
training_df = pd.concat(records, ignore_index=True)

training_df.to_parquet(OUTPUT_DATASET_PATH, index=False)

print("\n" + "=" * 60)
print("DATASET SAVED")
print("=" * 60)

print(f"Rows : {len(training_df)}")
print(f"Path : {OUTPUT_DATASET_PATH}")

BUILDING D2M TRAINING DATASET
Months to process : 400
Processed month : 198109
Processed month : 198110
Processed month : 198111
Processed month : 198112
Processed month : 198201
Processed month : 198202
Processed month : 198203
Processed month : 198204
Processed month : 198205
Processed month : 198206
Processed month : 198207
Processed month : 198208
Processed month : 198209
Processed month : 198210
Processed month : 198211
Processed month : 198212
Processed month : 198301
Processed month : 198302
Processed month : 198303
Processed month : 198304
Processed month : 198305
Processed month : 198306
Processed month : 198307
Processed month : 198308
Processed month : 198309
Processed month : 198310
Processed month : 198311
Processed month : 198312
Processed month : 198401
Processed month : 198402
Processed month : 198403
Processed month : 198404
Processed month : 198405
Processed month : 198406
Processed month : 198407
Processed month : 198408
Processed month : 198409
Processed month : 198

In [ ]:
"Make sure you correct the rasters before proceeding to the next cells!"

In [1]:
# ---------------------------------------------------------------------
# VALIDATION CELL
# This cell is intentionally self-contained.
# It is used only to validate corrected d2m rasters after:
#   1) model training
#   2) future raster correction
# ---------------------------------------------------------------------

from pathlib import Path
import rasterio
import numpy as np
import json

# ---------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought").resolve()

CORRECTED_ROOT = (
    PROJECT_ROOT
    / "rasters"
    / "corrected"
    / "cmip6"
)

SCENARIOS = ["ssp245", "ssp370", "ssp585"]

print("=" * 60)
print("CMIP6 CORRECTED D2M OUTPUT VALIDATION")
print("=" * 60)

results = []

for scenario in SCENARIOS:

    d2m_dir = CORRECTED_ROOT / scenario / "d2m"

    if not d2m_dir.exists():
        print(f"{scenario}: directory missing")
        continue

    tif_files = sorted(d2m_dir.glob("*.tif"))

    if not tif_files:
        print(f"{scenario}: no rasters found")
        continue

    sample_file = tif_files[0]

    with rasterio.open(sample_file) as src:

        arr = src.read(1)

        stats = {
            "scenario": scenario,
            "variable": "d2m",
            "status": "ok",
            "months_found": len(tif_files),
            "sample_month": sample_file.stem.split("_")[-1],
            "sample_file": sample_file.name,
            "crs": str(src.crs),
            "shape": (src.height, src.width),
            "min": float(np.nanmin(arr)),
            "max": float(np.nanmax(arr)),
            "mean": float(np.nanmean(arr)),
        }

        results.append(stats)

        print(stats)

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

CMIP6 CORRECTED D2M OUTPUT VALIDATION
{'scenario': 'ssp245', 'variable': 'd2m', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'd2m_corrected_202601.tif', 'crs': 'EPSG:4326', 'shape': (70, 80), 'min': 291.5008239746094, 'max': 292.7438049316406, 'mean': 292.5980529785156}
{'scenario': 'ssp370', 'variable': 'd2m', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'd2m_corrected_202601.tif', 'crs': 'EPSG:4326', 'shape': (70, 80), 'min': 287.2460021972656, 'max': 292.5643310546875, 'mean': 289.5967102050781}
{'scenario': 'ssp585', 'variable': 'd2m', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'd2m_corrected_202601.tif', 'crs': 'EPSG:4326', 'shape': (70, 80), 'min': 290.0360107421875, 'max': 292.1824645996094, 'mean': 290.9964904785156}

VALIDATION COMPLETE
